# 3주차 LLM API — 김예진 강사님 풀이 (Blind)
- 문제지만 보고 작성. 정답지 미참조.
- 10문제 전부 OpenAI API 호출 필수 → 코드 작성은 완료했으나 실제 실행은 본인 API 키가 있는 환경에서 수행 필요.

In [ ]:
!pip install openai rouge-score python-dotenv -q

In [ ]:
import os, time, json, re, math, itertools
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'

## Q1. TrackedLLMClient

In [1]:
class TrackedLLMClient:
    PRICE_PER_1K = {'gpt-4o-mini': {'input': 0.00015, 'output': 0.0006}}

    def __init__(self, model='gpt-4o-mini'):
        self.client = OpenAI()
        self.model = model
        self.call_log = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost = 0.0
        self.total_latency_ms = 0.0
        self.call_count = 0

    def generate(self, prompt, system_prompt=None, temperature=0.7, max_tokens=300):
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})

        last_err = None
        for attempt in range(1, 4):
            try:
                start = time.time()
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                latency_ms = (time.time() - start) * 1000
                u = response.usage
                price = self.PRICE_PER_1K[self.model]
                cost = (u.prompt_tokens / 1000) * price['input'] + (u.completion_tokens / 1000) * price['output']

                self.total_input_tokens += u.prompt_tokens
                self.total_output_tokens += u.completion_tokens
                self.total_cost += cost
                self.total_latency_ms += latency_ms
                self.call_count += 1

                log_entry = {
                    'model': self.model,
                    'input_tokens': u.prompt_tokens,
                    'output_tokens': u.completion_tokens,
                    'latency_ms': latency_ms,
                    'cost_usd': cost,
                    'status': 'success',
                    'attempts': attempt,
                }
                self.call_log.append(log_entry)
                return response.choices[0].message.content
            except Exception as e:
                last_err = e
                if attempt < 3:
                    time.sleep(2 ** (attempt - 1))  # 1s, 2s
                    continue
                self.call_log.append({
                    'model': self.model, 'status': 'failed',
                    'error': str(e), 'attempts': attempt,
                })
                raise

    def get_usage_summary(self):
        return {
            'call_count': self.call_count,
            'total_input_tokens': self.total_input_tokens,
            'total_output_tokens': self.total_output_tokens,
            'total_tokens': self.total_input_tokens + self.total_output_tokens,
            'total_cost_usd': round(self.total_cost, 6),
            'avg_latency_ms': round(self.total_latency_ms / max(self.call_count, 1), 1),
        }


tracker = TrackedLLMClient()
prompts = ['대한민국의 수도는?', '파이썬의 장점 3가지', '머신러닝을 한 문장으로']
for p in prompts:
    result = tracker.generate(p)
    print(f'Q: {p}\nA: {result[:80]}...\n')

summary = tracker.get_usage_summary()
print('=== 사용량 요약 ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

Q: 대한민국의 수도는?
A: 대한민국의 수도는 서울입니다....

Q: 파이썬의 장점 3가지
A: 파이썬의 장점은 다음과 같습니다:

1. **간결하고 읽기 쉬운 문법**: 파이썬은 코드의 가독성이 높아 개발자가 작성한 코드를 쉽게 이해할 수...

Q: 머신러닝을 한 문장으로
A: 머신러닝은 데이터로부터 학습하여 예측이나 결정을 자동으로 수행하는 인공지능의 한 분야입니다....

=== 사용량 요약 ===
  call_count: 3
  total_input_tokens: 46
  total_output_tokens: 260
  total_tokens: 306
  total_cost_usd: 0.000163
  avg_latency_ms: 2553.5


## Q2. Temperature 실험

In [2]:
def run_temperature_experiment(prompt, temperatures, n_trials=5):
    results = {}
    for temp in temperatures:
        responses = []
        for _ in range(n_trials):
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temp,
                max_tokens=150,
            )
            responses.append(r.choices[0].message.content.strip())

        unique_ratio = len(set(responses)) / len(responses)
        avg_length = sum(len(t) for t in responses) / len(responses)
        all_words = ' '.join(responses).split()
        vocab_diversity = (len(set(all_words)) / len(all_words)) if all_words else 0.0

        results[temp] = {
            'unique_ratio': unique_ratio,
            'avg_length': avg_length,
            'vocab_diversity': vocab_diversity,
            'responses': responses,
        }
    return results


prompt = '인공지능의 미래에 대해 한 문장으로 예측해주세요.'
temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]
results = run_temperature_experiment(prompt, temperatures)

print('=' * 65)
print(f"{'Temp':>6} | {'Unique Ratio':>12} | {'Avg Length':>10} | {'Vocab Div':>10}")
print('-' * 65)
for temp in temperatures:
    r = results[temp]
    print(f"{temp:>6.1f} | {r['unique_ratio']:>12.2f} | {r['avg_length']:>10.1f} | {r['vocab_diversity']:>10.3f}")

  Temp | Unique Ratio | Avg Length |  Vocab Div
-----------------------------------------------------------------
   0.0 |         0.40 |       63.6 |      0.292
   0.3 |         1.00 |       68.0 |      0.611
   0.7 |         1.00 |       74.4 |      0.643
   1.0 |         1.00 |       67.2 |      0.675
   1.5 |         1.00 |       76.6 |      0.820


## Q3. Top-p × Temperature 교차

In [3]:
def cross_experiment(prompt, temperatures, top_ps, n_trials=5):
    results = {}
    for temp, tp in itertools.product(temperatures, top_ps):
        responses, token_counts = [], []
        for _ in range(n_trials):
            r = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temp, top_p=tp, max_tokens=200,
            )
            responses.append(r.choices[0].message.content.strip())
            token_counts.append(r.usage.completion_tokens)

        all_words = ' '.join(responses).split()
        vocab_div = (len(set(all_words)) / len(all_words)) if all_words else 0.0
        results[(temp, tp)] = {
            'avg_tokens': sum(token_counts) / len(token_counts),
            'vocab_diversity': vocab_div,
            'responses': responses,
        }
    return results


prompt = '클라우드 컴퓨팅의 장단점을 설명해주세요.'
results = cross_experiment(prompt, [0.5, 1.0], [0.5, 0.9])

print('=' * 60)
print('[ 평균 토큰 수 ]')
print(f"{'':>15} | {'top_p=0.5':>12} | {'top_p=0.9':>12}")
print('-' * 60)
for temp in [0.5, 1.0]:
    v1 = results[(temp, 0.5)]['avg_tokens']
    v2 = results[(temp, 0.9)]['avg_tokens']
    print(f"{'temp='+str(temp):>15} | {v1:>12.1f} | {v2:>12.1f}")

print('\n[ 어휘 다양성 ]')
print(f"{'':>15} | {'top_p=0.5':>12} | {'top_p=0.9':>12}")
print('-' * 60)
for temp in [0.5, 1.0]:
    v1 = results[(temp, 0.5)]['vocab_diversity']
    v2 = results[(temp, 0.9)]['vocab_diversity']
    print(f"{'temp='+str(temp):>15} | {v1:>12.3f} | {v2:>12.3f}")

[ 평균 토큰 수 ]
                |    top_p=0.5 |    top_p=0.9
------------------------------------------------------------
       temp=0.5 |        200.0 |        200.0
       temp=1.0 |        200.0 |        200.0

[ 어휘 다양성 ]
                |    top_p=0.5 |    top_p=0.9
------------------------------------------------------------
       temp=0.5 |        0.299 |        0.351
       temp=1.0 |        0.319 |        0.419


## Q4. 스트리밍 + 금칙어 필터

In [4]:
def stream_with_filter(prompt, forbidden_words, max_tokens=500):
    start_time = time.time()
    first_token_time = None
    collected_text = ''
    token_count = 0
    stopped = False
    trigger = None

    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens, stream=True,
    )

    for chunk in stream:
        delta = chunk.choices[0].delta.content if chunk.choices else None
        if delta is None:
            continue
        if first_token_time is None:
            first_token_time = time.time()
        collected_text += delta
        token_count += 1
        hit = next((w for w in forbidden_words if w in collected_text), None)
        if hit:
            stopped = True
            trigger = hit
            break

    end_time = time.time()
    total_ms = (end_time - start_time) * 1000
    ttft_ms = ((first_token_time - start_time) * 1000) if first_token_time else 0.0
    gen_sec = (end_time - first_token_time) if first_token_time else 0.0
    tps = (token_count / gen_sec) if gen_sec > 0 else 0.0

    return {
        'text': collected_text,
        'stopped_by_filter': stopped,
        'trigger_word': trigger,
        'ttft_ms': ttft_ms,
        'total_ms': total_ms,
        'tokens_per_sec': tps,
        'token_count': token_count,
    }


r1 = stream_with_filter('대한민국의 계절별 특징을 설명해주세요.', ['폭력', '차별'])
print(f"[정상] 토큰: {r1['token_count']}, TTFT: {r1['ttft_ms']:.0f}ms, 속도: {r1['tokens_per_sec']:.1f} tok/s")
print(f"  텍스트: {r1['text'][:100]}...")

r2 = stream_with_filter('날씨와 기온에 대해 자세히 설명해주세요.', ['기온', '온도'])
print(f"\n[필터] 중단: {r2['stopped_by_filter']}, 감지어: '{r2['trigger_word']}'")
print(f"  텍스트: {r2['text'][:100]}...")

[정상] 토큰: 487, TTFT: 540ms, 속도: 69.2 tok/s
  텍스트: 대한민국은 사계절이 뚜렷한 기후 특성을 가지고 있습니다. 각 계절마다 고유한 특징이 있으며, 다음과 같은 특징들이 있습니다.

1. **봄 (3월 ~ 5월)**:
   - **온도...

[필터] 중단: True, 감지어: '기온'
  텍스트: 날씨와 기온...


## Q5. Dynamic Few-shot Selector

In [5]:
EXAMPLE_POOL = [
    ('이 영화 정말 감동적이었어요, 눈물이 멈추지 않았습니다.', 'POSITIVE'),
    ('연기력이 너무 떨어져서 보기 힘들었습니다.', 'NEGATIVE'),
    ('그저 그런 평범한 작품이었습니다.', 'NEUTRAL'),
    ('OST가 너무 좋아서 앨범을 구매했어요!', 'POSITIVE'),
    ('스토리가 억지스럽고 개연성이 없었어요.', 'NEGATIVE'),
    ('볼만은 하지만 특별한 점은 없었네요.', 'NEUTRAL'),
    ('최고의 명작! 두 번 세 번 봐도 질리지 않아요.', 'POSITIVE'),
    ('시간 낭비였습니다. 중간에 나올 뿐했어요.', 'NEGATIVE'),
    ('호불호가 갈릴 것 같은 작품입니다.', 'NEUTRAL'),
    ('배우들의 케미가 환상적이고 대사가 위트있어요.', 'POSITIVE'),
    ('결말이 너무 허무하고 실망스러웠습니다.', 'NEGATIVE'),
    ('기대 이하도 이상도 아닌 무난한 영화.', 'NEUTRAL'),
]


class DynamicFewShotSelector:
    def __init__(self, example_pool, k=4):
        self.examples = example_pool
        self.k = k

    def _compute_similarity(self, query, candidate):
        q = set(query.split())
        c = set(candidate.split())
        if not q or not c:
            return 0.0
        inter = q & c
        union = q | c
        return len(inter) / len(union)

    def _select_top_k(self, query):
        scored = [
            (self._compute_similarity(query, text), text, label)
            for text, label in self.examples
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [(t, l) for _, t, l in scored[: self.k]]

    def _apply_class_mixing(self, selected):
        # 연속 3개 같은 라벨 방지 — greedy 재배치
        remaining = list(selected)
        ordered = []
        while remaining:
            picked = None
            for i, (t, l) in enumerate(remaining):
                if len(ordered) >= 2 and ordered[-1][1] == l and ordered[-2][1] == l:
                    continue
                picked = i
                break
            if picked is None:
                picked = 0  # 강제 삽입
            ordered.append(remaining.pop(picked))
        return ordered

    def build_prompt(self, query):
        selected = self._select_top_k(query)
        mixed = self._apply_class_mixing(selected)
        header = '다음 리뷰의 감성을 POSITIVE, NEUTRAL, NEGATIVE 중 하나로 분류하세요.\n\n'
        body = ''.join(f'리뷰: {t}\n감성: {l}\n\n' for t, l in mixed)
        tail = f'리뷰: {query}\n감성:'
        return header + body + tail


selector = DynamicFewShotSelector(EXAMPLE_POOL, k=4)
test_query = '영상미는 좋았지만 내용이 공허해서 아쉬웠습니다.'
prompt = selector.build_prompt(test_query)
print(prompt)
print('\n' + '=' * 50)

result = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0,
)
print(f'분류 결과: {result.choices[0].message.content}')

다음 리뷰의 감성을 POSITIVE, NEUTRAL, NEGATIVE 중 하나로 분류하세요.

리뷰: 이 영화 정말 감동적이었어요, 눈물이 멈추지 않았습니다.
감성: POSITIVE

리뷰: 연기력이 너무 떨어져서 보기 힘들었습니다.
감성: NEGATIVE

리뷰: 그저 그런 평범한 작품이었습니다.
감성: NEUTRAL

리뷰: OST가 너무 좋아서 앨범을 구매했어요!
감성: POSITIVE

리뷰: 영상미는 좋았지만 내용이 공허해서 아쉬웠습니다.
감성:

분류 결과: NEGATIVE


## Q6. Stepback Prompting

In [6]:
def _chat(prompt, temperature=0.3, max_tokens=400):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature, max_tokens=max_tokens,
    )
    return r.choices[0].message.content.strip()


def stepback_pipeline(question):
    standard_answer = _chat(f'질문: {question}\n답변:')

    stepback_prompt = (
        '다음 질문의 답을 구하기 위해 먼저 알아야 할 상위 수준의 배경 질문을 하나만 생성하세요.\n'
        '원래 질문만 보고는 바로 답하기 어렵지만, 상위 질문의 답을 알면 원래 질문에 쉽게 답할 수 있어야 합니다.\n\n'
        f'원래 질문: {question}\n상위 배경 질문:'
    )
    stepback_question = _chat(stepback_prompt)
    stepback_answer = _chat(f'질문: {stepback_question}\n답변:')

    final_prompt = (
        f'다음 배경 지식을 참고하여 질문에 답하세요.\n\n[배경 지식]\n{stepback_answer}\n\n[질문]\n{question}\n\n[답변]'
    )
    final_answer = _chat(final_prompt)

    judge_prompt = (
        '두 답변을 비교 평가하세요. 각각 1-10점으로 채점하고 JSON으로만 응답하세요.\n\n'
        f'[질문] {question}\n\n[답변 A - Standard]\n{standard_answer}\n\n'
        f'[답변 B - Stepback]\n{final_answer}\n\n'
        'JSON 형식:\n{"score_A": <1-10>, "score_B": <1-10>, "winner": "A" 또는 "B", "reason": "<한줄 이유>"}'
    )
    raw = _chat(judge_prompt, temperature=0)
    # JSON 추출 (코드블록 제거 후 파싱)
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    try:
        judge_result = json.loads(m.group(0)) if m else {'raw': raw}
    except json.JSONDecodeError:
        judge_result = {'raw': raw}

    return {
        'stepback_question': stepback_question,
        'stepback_answer': stepback_answer,
        'final_answer': final_answer,
        'standard_answer': standard_answer,
        'judge_result': judge_result,
    }


questions = [
    '2024년 노벨 물리학상 수상자의 연구 분야가 AI 발전에 어떤 영향을 미쳄나요?',
    '한국의 출생률 저하가 2040년 부동산 시장에 미칠 영향은?',
]
for q in questions:
    r = stepback_pipeline(q)
    print(f'Q: {q}')
    print(f"  Stepback Q: {r['stepback_question']}")
    print(f"  Standard: {r['standard_answer'][:80]}...")
    print(f"  Stepback: {r['final_answer'][:80]}...")
    print(f"  Judge: {r['judge_result']}")
    print()

Q: 2024년 노벨 물리학상 수상자의 연구 분야가 AI 발전에 어떤 영향을 미쳤나요?
  Stepback Q: 2024년 노벨 물리학상 수상자가 연구한 주제는 무엇인가요?
  Standard: 2024년 노벨 물리학상 수상자의 연구 분야에 대한 구체적인 정보는 현재로서는 알 수 없지만, 일반적으로 물리학의 발전은 인공지능(AI) 기술에...
  Stepback: 현재로서는 2024년 노벨 물리학상 수상자의 연구 분야에 대한 정보가 없기 때문에 그들의 연구가 AI 발전에 어떤 영향을 미쳤는지 구체적으로 말...
  Judge: {'score_A': 8, 'score_B': 5, 'winner': 'A', 'reason': '답변 A는 물리학과 AI의 관계를 구체적으로 설명하며, 다양한 측면에서의 영향을 제시했다.'}

Q: 한국의 출생률 저하가 2040년 부동산 시장에 미칠 영향은?
  Stepback Q: 한국의 인구 구조 변화가 경제 전반에 미치는 영향은 무엇인가?
  Standard: 한국의 출생률 저하는 2040년 부동산 시장에 여러 가지 영향을 미칠 것으로 예상됩니다. 다음은 그 주요 영향들입니다.

1. **인구 감소**...
  Stepback: 한국의 출생률 저하는 2040년 부동산 시장에 여러 가지 중요한 영향을 미칠 것으로 예상됩니다. 주요 영향은 다음과 같습니다.

1. **주택 ...
  Judge: {'score_A': 8, 'score_B': 9, 'winner': 'B', 'reason': '답변 B가 더 구체적이고 구조적인 분석을 제공하며, 주택 시장의 변화에 대한 명확한 예시를 포함하고 있다.'}



## Q7. CoT + Self-Consistency

In [7]:
PROBLEMS = [
    {'question': '가게에서 사과 3개와 배 2개를 사면 4,100원이고, 사과 1개와 배 4개를 사면 5,300원입니다. 사과 1개의 가격은?', 'answer': '600'},
    {'question': 'A, B, C 세 사람이 달리기를 합니다. A는 B보다 빠르고, C는 A보다 느리지만 B보다 빠릅니다. 가장 느린 사람은?', 'answer': 'B'},
    {'question': '어떤 수에 5를 더한 후 3을 곱하면 45가 됩니다. 어떤 수는?', 'answer': '10'},
]

COT_DEMOS = """Q: 연필 2자루와 지우개 3개를 사면 1,300원이고, 연필 4자루와 지우개 1개를 사면 1,500원입니다. 연필 1자루의 가격은?
A: 연립방정식을 세운다.
   2x + 3y = 1300 ... (1)
   4x + y = 1500 ... (2)
   (2)에서 y = 1500 - 4x → (1)에 대입
   2x + 3(1500-4x) = 1300 → 2x + 4500 - 12x = 1300 → -10x = -3200 → x = 320
   ### 최종 정답: 320

Q: X는 Y보다 키가 크고, Z는 X보다 작지만 Y보다 큽니다. 키가 가장 작은 사람은?
A: 조건 정리: X > Y, X > Z > Y. 따라서 Y가 가장 작습니다.
   ### 최종 정답: Y
"""


def extract_answer(text):
    m = re.search(r'###\s*최종 정답\s*:\s*([^\n]+)', text)
    if m:
        return m.group(1).strip().rstrip('.')
    # fallback — 마지막 줄의 영숫자
    last_line = text.strip().splitlines()[-1] if text.strip() else ''
    m2 = re.search(r'[A-Za-z0-9가-힣]+', last_line)
    return m2.group(0) if m2 else ''


def zero_shot_cot(question):
    prompt = (
        f'Q: {question}\n단계별로 생각해봅시다.\n'
        '마지막 줄에 반드시 "### 최종 정답: <답>" 형식으로 답하세요.\n\nA:'
    )
    r = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=400,
    )
    return r.choices[0].message.content


def few_shot_cot(question):
    prompt = COT_DEMOS + f'\nQ: {question}\nA:'
    r = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}],
        temperature=0, max_tokens=400,
    )
    return r.choices[0].message.content


def self_consistency(question, n=7, temperature=0.8):
    answers = []
    prompt = COT_DEMOS + f'\nQ: {question}\nA:'
    for _ in range(n):
        r = client.chat.completions.create(
            model=MODEL, messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature, max_tokens=400,
        )
        answers.append(extract_answer(r.choices[0].message.content))
    vote = Counter(answers)
    final_answer, final_count = vote.most_common(1)[0]
    return {
        'final_answer': final_answer,
        'confidence': final_count / len(answers),
        'all_answers': answers,
        'vote_counts': dict(vote),
    }


print('=' * 70)
print(f"{'문제':>4} | {'정답':>4} | {'Zero-CoT':>10} | {'Few-CoT':>10} | {'SC(N=7)':>10} | {'SC 신뢰도':>8}")
print('-' * 70)
for i, prob in enumerate(PROBLEMS, 1):
    zc = extract_answer(zero_shot_cot(prob['question']))
    fc = extract_answer(few_shot_cot(prob['question']))
    sc = self_consistency(prob['question'])
    print(f"  {i:>2} | {prob['answer']:>4} | {zc:>10} | {fc:>10} | {sc['final_answer']:>10} | {sc['confidence']:>7.0%}")

  문제 |   정답 |   Zero-CoT |    Few-CoT |    SC(N=7) |   SC 신뢰도
----------------------------------------------------------------------
   1 |  600 |        580 |      580 원 |        580 |     43%
   2 |    B |          C |          C |          C |    100%
   3 |   10 |         10 |         10 |         10 |     86%


## Q8 (킬러). Tree of Thoughts

In [8]:
RESOURCE_PROBLEM = """
당신은 스타트업 CTO입니다. 다음 분기에 개발팀 6명을 3개 프로젝트에 배분해야 합니다.
- 프로젝트 A (신규 기능): 매출 기여 높음, 최소 2명 필요
- 프로젝트 B (기술 부채): 장기 안정성, 최소 1명 필요
- 프로젝트 C (보안 패치): 긴급도 높음, 최소 1명 필요
제약: 총 6명, 각 프로젝트 최소 인원 충족, 분기 내 완료 가능해야 함
최적의 인원 배분과 근거를 제시하세요.
"""


def _chat_gen(prompt, temperature=0.9, max_tokens=300):
    r = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature, max_tokens=max_tokens,
    )
    return r.choices[0].message.content.strip()


def tree_of_thoughts(problem, n_initial=3, n_expand=2):
    gen_prompt = (
        f'{problem}\n\n위 문제에 대해 서로 다른 접근법을 하나만 제안하세요. '
        '인원 배분(A:?명, B:?명, C:?명)과 핵심 논리를 2-3줄로 작성하세요.'
    )
    strategies = [_chat_gen(gen_prompt, temperature=0.9, max_tokens=200) for _ in range(n_initial)]

    def evaluate_strategy(strategy):
        eval_prompt = (
            '다음 전략을 1-10점으로 평가하세요.\n'
            '평가 기준: 실현 가능성, 리스크 관리, 매출 기여도\n'
            '반드시 "점수: N" 형식으로만 답하세요.\n\n'
            f'전략: {strategy}\n'
        )
        raw = _chat_gen(eval_prompt, temperature=0, max_tokens=50)
        m = re.search(r'\d+', raw)
        return int(m.group(0)) if m else 0

    scores = [evaluate_strategy(s) for s in strategies]

    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n_expand]
    expanded = []
    for idx in top_indices:
        expand_prompt = (
            '다음 전략을 더 구체적으로 발전시키세요. '
            '주차별 실행 계획과 리스크 대응 방안을 추가하세요.\n\n'
            f'전략: {strategies[idx]}\n확장된 전략:'
        )
        expanded.append(_chat_gen(expand_prompt, temperature=0.5, max_tokens=500))

    final_scores = [evaluate_strategy(e) for e in expanded]
    best_idx = max(range(len(final_scores)), key=lambda i: final_scores[i])

    return {
        'initial_strategies': strategies,
        'initial_scores': scores,
        'expanded_strategies': expanded,
        'final_scores': final_scores,
        'best_strategy': expanded[best_idx],
        'best_score': final_scores[best_idx],
    }


result = tree_of_thoughts(RESOURCE_PROBLEM)
print('=== Tree of Thoughts 결과 ===')
for i, (s, sc) in enumerate(zip(result['initial_strategies'], result['initial_scores'])):
    print(f'\n[초기 전략 {i+1}] (점수: {sc})')
    print(f'  {s[:100]}...')
print(f"\n🏆 최종 선택 (점수: {result['best_score']}):")
print(result['best_strategy'])

=== Tree of Thoughts 결과 ===

[초기 전략 1] (점수: 8)
  인원 배분: A: 3명, B: 1명, C: 2명

핵심 논리: 프로젝트 A는 매출 기여가 높기 때문에 우선 3명을 배정하여 기능 개발을 최대화하고, 프로젝트 C는 긴급성이 높으므로...

[초기 전략 2] (점수: 8)
  인원 배분: A: 3명, B: 1명, C: 2명

핵심 논리: 프로젝트 A는 매출 기여도가 높으며 신규 기능 개발에 집중해야 하므로 3명을 배정하여 강력한 개발력을 확보합니다. 프...

[초기 전략 3] (점수: 8)
  인원 배분: A: 3명, B: 1명, C: 2명

핵심 논리: 프로젝트 A는 매출 기여도가 가장 높기 때문에 최대한 많은 인원을 배분하여 빠른 시간 내에 새로운 기능을 론칭해야 합...

🏆 최종 선택 (점수: 8):
### 확장된 전략

#### 인원 배분
- **프로젝트 A**: 3명
- **프로젝트 B**: 1명
- **프로젝트 C**: 2명

#### 핵심 논리
- **프로젝트 A**: 매출 기여가 높아 기능 개발을 최대화하기 위해 3명 배정.
- **프로젝트 B**: 기술 부채 해결을 위한 안정성을 위해 1명 배정. 최소 요구사항 충족 가능.
- **프로젝트 C**: 긴급한 보안 패치 진행을 위해 2명 배정.

### 주차별 실행 계획

#### 1주차
- **프로젝트 A**
  - 기능 요구사항 분석 및 우선순위 설정 (3명)
  - 초기 프로토타입 개발 시작
- **프로젝트 B**
  - 기술 부채 목록 작성 및 우선순위 설정 (1명)
- **프로젝트 C**
  - 보안 패치 요구사항 분석 (2명)
  - 긴급 패치 계획 수립

#### 2주차
- **프로젝트 A**
  - 프로토타입 개발 완료 및 내부 테스트 시작 (3명)
- **프로젝트 B**
  - 기술 부채 해결을 위한 코드 리팩토링 시작 (1명)
- **프로젝트 C**
  - 보안 패치 개발 시작 (2명)

#### 3주차
- **프로젝트 A**
  - 내부 테스

## Q9 (킬러). ReAct Agent

In [9]:
def tool_search(query):
    kb = {
        '대한민국 인구': '대한민국의 2024년 추정 인구는 약 5,175만 명입니다.',
        '서울 면적': '서울의 면적은 약 605.2 km²입니다.',
        '한강 길이': '한강의 총 길이는 약 514km입니다.',
        '에펄탑 높이': '에펄탑의 높이는 약 330m입니다.',
        '빛의 속도': '빛의 속도는 약 299,792,458 m/s입니다.',
    }
    for key, value in kb.items():
        if any(word in query for word in key.split()):
            return value
    return f"'{query}'에 대한 정보를 찾을 수 없습니다."


def tool_calculate(expression):
    try:
        allowed = {'__builtins__': {}, 'math': math}
        result = eval(expression, allowed)
        return f'계산 결과: {result}'
    except Exception as e:
        return f'계산 오류: {e}'


def tool_lookup_date(query):
    from datetime import datetime
    if '오늘' in query or '현재' in query:
        return f"현재 날짜: {datetime.now().strftime('%Y년 %m월 %d일')}"
    return '날짜 정보를 찾을 수 없습니다.'


TOOLS = {'search': tool_search, 'calculate': tool_calculate, 'lookup_date': tool_lookup_date}

REACT_SYSTEM_PROMPT = """당신은 도구를 사용하여 질문에 답하는 AI 에이전트입니다.

사용 가능한 도구:
- search(query): 지식 검색
- calculate(expression): 수학 계산 (파이썬 수식)
- lookup_date(query): 날짜/시간 조회

반드시 아래 형식을 따르세요:
Thought: (현재 상황 분석과 다음 행동 계획)
Action: tool_name(argument)

또는 최종 답변이 준비되면:
Thought: (최종 정리)
Final Answer: (최종 답변)"""


def parse_action(text):
    m = re.search(r'Action:\s*([A-Za-z_][A-Za-z0-9_]*)\((.*)\)\s*$', text, re.MULTILINE)
    if not m:
        return None
    tool_name = m.group(1)
    argument = m.group(2).strip().strip('\'"')
    return (tool_name, argument)


def react_agent(question, max_steps=5):
    messages = [
        {'role': 'system', 'content': REACT_SYSTEM_PROMPT},
        {'role': 'user', 'content': f'질문: {question}'},
    ]
    trajectory = []

    for step in range(max_steps):
        r = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0.2, max_tokens=300,
        )
        response_text = r.choices[0].message.content
        trajectory.append({'step': step + 1, 'agent': response_text})

        if 'Final Answer:' in response_text:
            final = response_text.split('Final Answer:')[-1].strip()
            return {'answer': final, 'trajectory': trajectory}

        action = parse_action(response_text)
        if action is None:
            messages.append({'role': 'assistant', 'content': response_text})
            messages.append({'role': 'user', 'content': '올바른 Action 형식을 사용해주세요: Action: tool_name(argument)'})
            continue

        tool_name, argument = action
        if tool_name in TOOLS:
            observation = TOOLS[tool_name](argument)
        else:
            observation = f'알 수 없는 도구: {tool_name}'

        trajectory.append({'step': step + 1, 'observation': observation})
        messages.append({'role': 'assistant', 'content': response_text})
        messages.append({'role': 'user', 'content': f'Observation: {observation}'})

    return {'answer': '최대 스텝 초과', 'trajectory': trajectory}


test_questions = [
    '서울의 인구 밀도(명/km²)는 얼마인가요?',
    '에펄탑 높이의 3배는 몇 미터이고, 한강 길이와 비교하면 어떤가요?',
]
for q in test_questions:
    print(f'Q: {q}')
    result = react_agent(q)
    print(f"A: {result['answer']}")
    print(f"Steps: {len(result['trajectory'])}")
    for t in result['trajectory']:
        if 'agent' in t:
            print(f"  [Agent Step {t['step']}] {t['agent'][:120]}...")
        if 'observation' in t:
            print(f"  [Observation] {t['observation']}")
    print()

Q: 서울의 인구 밀도(명/km²)는 얼마인가요?
A: 최대 스텝 초과
Steps: 10
  [Agent Step 1] Thought: 서울의 인구 밀도를 알아보기 위해 최신 데이터를 검색해야 합니다. 
Action: search("서울 인구 밀도 2023")...
  [Observation] 대한민국의 2024년 추정 인구는 약 5,175만 명입니다.
  [Agent Step 2] Thought: 대한민국의 전체 인구가 주어졌지만, 서울의 인구 밀도를 계산하기 위해서는 서울의 인구와 면적이 필요합니다. 서울의 인구와 면적에 대한 정보를 검색하겠습니다. 
Action: search("서울 인구 ...
  [Observation] 대한민국의 2024년 추정 인구는 약 5,175만 명입니다.
  [Agent Step 3] Thought: 서울의 인구에 대한 정보가 필요하지만, 현재 제공된 정보는 대한민국 전체 인구에 대한 것입니다. 서울의 인구와 면적을 확인하기 위해 추가 검색이 필요합니다. 
Action: search("서울 면적"...
  [Observation] 서울의 면적은 약 605.2 km²입니다.
  [Agent Step 4] Thought: 서울의 면적이 605.2 km²로 확인되었습니다. 이제 서울의 인구를 알아내어 인구 밀도를 계산할 수 있습니다. 서울의 인구를 검색하겠습니다. 
Action: search("서울 인구 2023")...
  [Observation] 대한민국의 2024년 추정 인구는 약 5,175만 명입니다.
  [Agent Step 5] Thought: 서울의 인구에 대한 정보가 여전히 부족합니다. 서울의 인구를 직접적으로 검색하여 인구 밀도를 계산해야 합니다. 
Action: search("서울 2023년 인구")...
  [Observation] 대한민국의 2024년 추정 인구는 약 5,175만 명입니다.

Q: 에펄탑 높이의 3배는 몇 미터이고, 한강 길이와 비교하면 어떤가요?
A: 에펠탑 높이의 3배는 900미터이며, 이는

## Q10 (킬러). PromptBenchmark

In [10]:
from rouge_score import rouge_scorer

TEST_CASES = [
    {'question': '머신러닝과 딥러닝의 차이점을 설명하세요.',
     'reference': '머신러닝은 데이터에서 패턴을 학습하는 알고리즘의 총칭이고, 딥러닝은 심층 신경망을 사용하는 머신러닝의 하위 분야입니다.'},
    {'question': 'API와 SDK의 차이를 설명하세요.',
     'reference': 'API는 소프트웨어 간 통신 규약이고, SDK는 API를 포함한 개발 도구 모음입니다.'},
    {'question': '캐시(Cache)의 역할을 설명하세요.',
     'reference': '캐시는 자주 사용되는 데이터를 빠른 저장소에 임시 보관하여 접근 속도를 높이는 메커니즘입니다.'},
    {'question': 'REST API의 핵심 원칙을 설명하세요.',
     'reference': 'REST는 무상태성, 자원 기반 URI, HTTP 메서드 활용, 표준 응답 코드 사용을 핵심 원칙으로 합니다.'},
    {'question': 'Docker 컨테이너와 가상머신의 차이를 설명하세요.',
     'reference': '컨테이너는 OS 커널을 공유하여 가볍고 빠르며, 가상머신은 전체 OS를 포함하여 무겁지만 완전한 격리를 제공합니다.'},
]

PRICE = {'input': 0.00015, 'output': 0.0006}


class PromptBenchmark:
    def __init__(self, test_cases):
        self.test_cases = test_cases
        self.scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
        self.results = {}
        self.strategy_cost = {}

    def _call(self, prompt, temperature=0.3, max_tokens=300, strategy=None):
        r = client.chat.completions.create(
            model=MODEL, messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature, max_tokens=max_tokens,
        )
        u = r.usage
        cost = (u.prompt_tokens / 1000) * PRICE['input'] + (u.completion_tokens / 1000) * PRICE['output']
        if strategy is not None:
            self.strategy_cost[strategy] = self.strategy_cost.get(strategy, 0.0) + cost
        return r.choices[0].message.content.strip()

    def _zero_shot(self, question):
        return self._call(f'다음 질문에 간결하게 답하세요.\n\n질문: {question}\n답변:', strategy='Zero-shot')

    def _few_shot(self, question):
        examples = [tc for tc in self.test_cases if tc['question'] != question][:2]
        prompt = '다음 형식으로 질문에 간결히 답하세요.\n\n'
        for ex in examples:
            prompt += f"질문: {ex['question']}\n답변: {ex['reference']}\n\n"
        prompt += f'질문: {question}\n답변:'
        return self._call(prompt, strategy='Few-shot')

    def _cot(self, question):
        return self._call(f'단계별로 생각하여 다음 질문에 답하세요.\n\n질문: {question}\n답변:', strategy='CoT')

    def _self_consistency(self, question, n=5):
        candidates = [
            self._call(f'단계별로 생각하여 다음 질문에 답하세요.\n\n질문: {question}\n답변:',
                       temperature=0.7, strategy='Self-Consistency')
            for _ in range(n)
        ]
        return max(candidates, key=len)

    def _compute_rouge_l(self, reference, candidate):
        return self.scorer.score(reference, candidate)['rougeL'].fmeasure

    def _llm_judge(self, question, reference, candidate):
        prompt = (
            '다음 질문에 대한 참조 답변과 후보 답변을 비교하여 1-10점으로 평가하세요. 숫자만 응답하세요.\n\n'
            f'질문: {question}\n참조 답변: {reference}\n후보 답변: {candidate}\n점수:'
        )
        raw = self._call(prompt, temperature=0, max_tokens=10)  # Judge 비용은 별도 수집 X (strategy=None)
        m = re.search(r'\d+', raw)
        return int(m.group(0)) if m else 0

    def run_benchmark(self):
        strategies = {
            'Zero-shot': self._zero_shot,
            'Few-shot': self._few_shot,
            'CoT': self._cot,
            'Self-Consistency': self._self_consistency,
        }
        for name, func in strategies.items():
            rouges, judges = [], []
            for tc in self.test_cases:
                ans = func(tc['question'])
                rouges.append(self._compute_rouge_l(tc['reference'], ans))
                judges.append(self._llm_judge(tc['question'], tc['reference'], ans))
            avg_r = sum(rouges) / len(rouges)
            avg_j = sum(judges) / len(judges)
            combined = (avg_r + avg_j / 10) / 2
            cost = self.strategy_cost.get(name, 0.0)
            self.results[name] = {
                'avg_rouge_l': avg_r, 'avg_judge': avg_j, 'combined': combined,
                'cost_usd': cost,
                'quality_per_dollar': (combined / cost) if cost > 0 else 0.0,
            }
        return self.results

    def print_leaderboard(self):
        ranked = sorted(self.results.items(), key=lambda kv: kv[1]['quality_per_dollar'], reverse=True)
        print(' Rank | Strategy         | ROUGE-L | Judge | Combined | Cost($)  | Quality/$')
        print('-' * 85)
        for rank, (name, r) in enumerate(ranked, 1):
            print(f"  {rank:>3} | {name:<16} | {r['avg_rouge_l']:>7.3f} | {r['avg_judge']:>5.2f} | "
                  f"{r['combined']:>8.3f} | {r['cost_usd']:>7.5f} | {r['quality_per_dollar']:>9.1f}")
        print(f"\n추천 전략 (Quality/$ 1위): {ranked[0][0]}")


bench = PromptBenchmark(TEST_CASES)
results = bench.run_benchmark()
bench.print_leaderboard()

 Rank | Strategy         | ROUGE-L | Judge | Combined | Cost($)  | Quality/$
-------------------------------------------------------------------------------------
    1 | Few-shot         |   0.280 |  8.40 |    0.560 | 0.00023 |    2475.7
    2 | Zero-shot        |   0.214 |  8.60 |    0.537 | 0.00037 |    1449.2
    3 | CoT              |   0.069 |  8.00 |    0.434 | 0.00093 |     468.1
    4 | Self-Consistency |   0.061 |  8.20 |    0.441 | 0.00464 |      95.0

추천 전략 (Quality/$ 1위): Few-shot
